## 0. Import & Config

In [4]:
import warnings
warnings.filterwarnings("ignore")

try:
    import koreanize_matplotlib
except ImportError:
    pass
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import optuna
from exp.my_ml_kit import *

RANDOM_STATE = 42
TARGET = "임신 성공 여부"
ID_COL = "ID"
TEST_SIZE = 0.2

TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"
SUBMISSION_PATH = "../data/sample_submission.csv"

NA_VALUES = ["None", "none", "NULL", "null","nan"]


## 1. 데이터 로드


In [5]:
train_data = pd.read_csv(TRAIN_PATH, na_values=NA_VALUES)
test_data = pd.read_csv(TEST_PATH, na_values=NA_VALUES)
submission_data = pd.read_csv(SUBMISSION_PATH)

print("train:", train_data.shape)
print("test :", test_data.shape)
print("submission:", submission_data.shape)

display(train_data.head())


train: (256351, 69)
test : (90067, 68)
submission: (90067, 2)


,ID,시술 시기 코드,시술 당시 나이,임신 시도 또는 마지막 임신 경과 연수,시술 유형,특정 시술 유형,배란 자극 여부,배란 유도 유형,단일 배아 이식 여부,착상 전 유전 검사 사용 여부,...,기증 배아 사용 여부,대리모 여부,PGD 시술 여부,PGS 시술 여부,난자 채취 경과일,난자 해동 경과일,난자 혼합 경과일,배아 이식 경과일,배아 해동 경과일,임신 성공 여부
0,TRAIN_000000,TRZKPL,만18-34세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,3.0,NaN,0
1,TRAIN_000001,TRYBLT,만45-50세,NaN,IVF,ICSI,0,알 수 없음,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,NaN,NaN,0
2,TRAIN_000002,TRVNRY,만18-34세,NaN,IVF,IVF,1,기록되지 않은 시행,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,2.0,NaN,0
3,TRAIN_000003,TRJXFG,만35-37세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,NaN,NaN,0
4,TRAIN_000004,TRVNRY,만18-34세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,3.0,NaN,0


## 데이터 전처리 함수

In [6]:
def data_preprocessing(df):
    df = df.copy()
    time_cols = [
        '임신 시도 또는 마지막 임신 경과 연수',
        '난자 해동 경과일',
        '난자 혼합 경과일',
        '배아 이식 경과일',
        '배아 해동 경과일'
    ]

    for col in time_cols:
        df[f'{col}_performed'] = (
            df[col].notnull()
        ).astype(int)


    df = df.fillna(0)
    infertility_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 남성 요인',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    male_cols = [
        '불임 원인 - 남성 요인',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    female_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증'
    ]

    df['불임원인_총개수'] = df[infertility_cols].sum(axis=1)

    df['남성_원인_수'] = df[male_cols].sum(axis=1)

    df['여성_원인_수'] = df[female_cols].sum(axis=1)

    df['남녀_복합_원인'] = (
        (df['남성_원인_수'] > 0) &
        (df['여성_원인_수'] > 0)
    ).astype(int)

    df['원인불명'] = (
        df['불임원인_총개수'] == 0
    ).astype(int)

    count_cols = [
        '총 시술 횟수',
        'IVF 시술 횟수',
        'DI 시술 횟수',
        '총 임신 횟수',
        'IVF 임신 횟수',
        'DI 임신 횟수',
        '총 출산 횟수',
        'IVF 출산 횟수',
        'DI 출산 횟수',
        '클리닉 내 총 시술 횟수'
    ]

    count_map = {
        '0회': 0,
        '1회': 1,
        '2회': 2,
        '3회': 3,
        '4회': 4,
        '5회': 5,
        '6회 이상': 6,
    }

    for col in count_cols:
        df[col] = df[col].map(count_map).astype(float)

    df['고령여부'] = df['시술 당시 나이'].isin([
        '만38-39세',
        '만40-42세',
        '만43-44세',
        '만45-50세'
    ]).astype(int)


    df['배아_생성률'] = np.where(
        df['혼합된 난자 수'] == 0,
        0,
        df['총 생성 배아 수'] / df['혼합된 난자 수']
    )

    # 2. 배아 이식 효율
    df['배아_이식률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['이식된 배아 수'] / df['총 생성 배아 수']
    )

    # 3. 배아 냉동 비율
    df['배아_냉동률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['저장된 배아 수'] / df['총 생성 배아 수']
    )
    df['IVF_임신성공률'] = np.where(
        df['IVF 시술 횟수'] == 0,
        0,
        df['IVF 임신 횟수'] / df['IVF 시술 횟수']
    )

    df['DI_임신성공률'] = np.where(
        df['DI 시술 횟수'] == 0,
        0,
        df['DI 임신 횟수'] / df['DI 시술 횟수']
    )

    df['고령_난자수_interaction'] = (
        df['고령여부'] *
        df['수집된 신선 난자 수']
    )

    df['배아이식_수행여부'] = (
        df['이식된 배아 수'] > 0
    ).astype(int)

    df['배아_이식_집중도'] = np.where(
        (df['이식된 배아 수'] + df['저장된 배아 수']) == 0,
        0,
        df['이식된 배아 수'] /
        (
            df['이식된 배아 수'] +
            df['저장된 배아 수']
        )
    )
    df["배아 생성 주요 이유"] = (
        df["배아 생성 주요 이유"]
        .astype(str)
        .astype("category")
    )
    df["특정 시술 유형"] = (
        df["특정 시술 유형"]
        .astype(str)
        .astype("category")
    )

    # 고령 × 이식 배아 수
    df['고령_배아이식'] = (
        df['고령여부'] *
        df['이식된 배아 수']
    )

    # 고령 × 총 생성 배아 수
    df['고령_배아생성'] = (
        df['고령여부'] *
        df['총 생성 배아 수']
    )

    # 고령 × 저장 배아 수
    df['고령_배아저장'] = (
        df['고령여부'] *
        df['저장된 배아 수']
    )

    # 고령 × 미세주입 난자 수
    df['고령_미세주입난자'] = (
        df['고령여부'] *
        df['미세주입된 난자 수']
    )

    df['출산_임신_전환율'] = np.where(
        df['총 임신 횟수'] == 0,
        0,
        df['총 출산 횟수'] / df['총 임신 횟수']
    )

    df['클리닉_집중도'] = np.where(
        df['총 시술 횟수'] == 0,
        0,
        df['클리닉 내 총 시술 횟수'] / df['총 시술 횟수']
    )

    df['첫_시술_여부'] = (
        df['총 시술 횟수'] == 0
    ).astype(int)



    binary_keywords = [
        "코드", "나이", "유형", "여부", "원인", "이유", "횟수", "출처"
    ]

    binary_cols = [
        col for col in df.columns
        if any(keyword in col for keyword in binary_keywords)
    ]

    df[binary_cols] = df[binary_cols].astype('category')

    object_cols = df.select_dtypes(include="object").columns

    df[object_cols] = df[object_cols].astype(str)

    cat_cols = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        if col != TARGET:
            df[col] = df[col].astype(str)

    drop_cols = [
        '배아이식_수행여부'
    ]
    df = df.drop(
        columns=drop_cols
        )
    return df

In [7]:
train_data.select_dtypes("object").columns

Index(['ID', '시술 시기 코드', '시술 당시 나이', '시술 유형', '특정 시술 유형', '배란 유도 유형',
       '배아 생성 주요 이유', '총 시술 횟수', '클리닉 내 총 시술 횟수', 'IVF 시술 횟수', 'DI 시술 횟수',
       '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수', '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수',
       '난자 출처', '정자 출처', '난자 기증자 나이', '정자 기증자 나이'],
      dtype='str')

In [7]:
train_data_processed = data_preprocessing(train_data)
test_data_processed = data_preprocessing(test_data)

X = train_data_processed.drop(columns=[ID_COL, TARGET])
y = train_data_processed[TARGET]

X_test = test_data_processed.drop(columns=[ID_COL])

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)
print(X_train.shape, X_val.shape)
print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))


(205080, 92) (51271, 92)
임신 성공 여부
0    0.741652
1    0.258348
Name: proportion, dtype: float64
임신 성공 여부
0    0.741647
1    0.258353
Name: proportion, dtype: float64


In [8]:
numeric_features = X.select_dtypes(
        include=np.number
        ).columns.tolist()

categorical_features = X.select_dtypes(
    include=["category", "object","str"]
    ).columns.tolist()

numeric_transformer = Pipeline([
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

In [21]:
cat_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    class_weights=[1, 190123 / 66228]
)

cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50
)

y_val_pred = cat_model.predict(X_val)
y_val_proba = cat_model.predict_proba(X_val)[:, 1]

print("f1:", f1_score(y_val, y_val_pred))
print("precision:", precision_score(y_val, y_val_pred))
print("recall:", recall_score(y_val, y_val_pred))
print("roc_auc:", roc_auc_score(y_val, y_val_proba))




0:	test: 0.7187693	best: 0.7187693 (0)	total: 2.17s	remaining: 36m 7s
100:	test: 0.7342821	best: 0.7343048 (99)	total: 44.1s	remaining: 6m 32s
200:	test: 0.7358076	best: 0.7358131 (199)	total: 1m 23s	remaining: 5m 31s
300:	test: 0.7365656	best: 0.7365721 (299)	total: 1m 59s	remaining: 4m 37s
400:	test: 0.7368373	best: 0.7368978 (380)	total: 2m 31s	remaining: 3m 46s
500:	test: 0.7370993	best: 0.7371045 (492)	total: 3m 9s	remaining: 3m 9s
600:	test: 0.7371639	best: 0.7371711 (594)	total: 3m 48s	remaining: 2m 31s
700:	test: 0.7372183	best: 0.7372703 (679)	total: 4m 24s	remaining: 1m 52s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7372703168
bestIteration = 679

Shrink model to first 680 iterations.
f1: 0.5146154999250487
precision: 0.3845780433159074
recall: 0.777517741204892
roc_auc: 0.7372703168276867


## FI

In [6]:
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": cat_model.get_feature_importance()
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

print(importance_df.head(30))

                feature  importance
41             이식된 배아 수   30.211822
74            배아_이식_집중도   26.579933
47          수집된 신선 난자 수    5.298930
52                난자 출처    4.514083
73   고령_난자수_interaction    3.696788
77               배아_냉동률    3.534816
1              시술 당시 나이    3.238554
70  배아 이식 경과일_performed    2.724797
38            총 생성 배아 수    2.680301
65            배아 이식 경과일    2.305180
6              배란 유도 유형    1.409667
53                정자 출처    1.371343
43             저장된 배아 수    1.273345
57          신선 배아 사용 여부    0.989283
75               배아_생성률    0.719379
27          배아 생성 주요 이유    0.573660
45             해동된 배아 수    0.560129
72                 고령여부    0.558019
76               배아_이식률    0.541363
0              시술 시기 코드    0.525854
4              특정 시술 유형    0.501040
31             DI 시술 횟수    0.463616
49             혼합된 난자 수    0.434921
78              고령_배아이식    0.422050
30            IVF 시술 횟수    0.393998
32              총 임신 횟수    0.370055
54            난자 기증자 나이    0

## k-fold CV

In [119]:
X_cb = X.copy()
y_cb = y.astype(int)

cat_cols = X_cb.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols:
    X_cb[col] = X_cb[col].astype(str)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_proba = np.zeros(len(X_cb))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_cb, y_cb), 1):
    X_train_fold = X_cb.iloc[train_idx].copy()
    X_val_fold = X_cb.iloc[val_idx].copy()
    y_train_fold = y_cb.iloc[train_idx]
    y_val_fold = y_cb.iloc[val_idx]

    cat_model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=100,
        class_weights=[1, 190123 / 66228]
    )

    cat_model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=cat_cols,
        eval_set=(X_val_fold, y_val_fold),
        early_stopping_rounds=50
    )

    val_proba = cat_model.predict_proba(X_val_fold)[:, 1]
    oof_proba[val_idx] = val_proba

    fold_auc = roc_auc_score(y_val_fold, val_proba)
    fold_scores.append(fold_auc)

    print(f"Fold {fold} ROC-AUC: {fold_auc:.6f}")

print("\n====================")
print("Fold scores:", fold_scores)
print("Mean ROC-AUC:", np.mean(fold_scores))
print("Std ROC-AUC:", np.std(fold_scores))
print("OOF ROC-AUC:", roc_auc_score(y_cb, oof_proba))

0:	test: 0.7174635	best: 0.7174635 (0)	total: 4.2s	remaining: 1h 10m
100:	test: 0.7357816	best: 0.7357816 (100)	total: 15.8s	remaining: 2m 20s
200:	test: 0.7372074	best: 0.7372074 (200)	total: 43.5s	remaining: 2m 52s
300:	test: 0.7377213	best: 0.7377213 (300)	total: 1m 31s	remaining: 3m 32s
400:	test: 0.7378730	best: 0.7378730 (400)	total: 2m 4s	remaining: 3m 5s
500:	test: 0.7379374	best: 0.7379587 (458)	total: 2m 31s	remaining: 2m 31s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7379586667
bestIteration = 458

Shrink model to first 459 iterations.
Fold 1 ROC-AUC: 0.737959
0:	test: 0.7251134	best: 0.7251134 (0)	total: 4.57s	remaining: 1h 16m 1s
100:	test: 0.7394004	best: 0.7394004 (100)	total: 48s	remaining: 7m 7s
200:	test: 0.7414647	best: 0.7414647 (200)	total: 1m 21s	remaining: 5m 24s
300:	test: 0.7423987	best: 0.7424127 (298)	total: 2m 11s	remaining: 5m 5s
400:	test: 0.7426357	best: 0.7426392 (394)	total: 2m 58s	remaining: 4m 26s
500:	test: 0.7428376	best: 0.

## OPTUNA

In [9]:
cat_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols:
    X_train[col] = X_train[col].astype(str)
    X_val[col] = X_val[col].astype(str)

def objective(trial):
    params = {
        "iterations": 1500,
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.08, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 20, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 5),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": 42,
        "verbose": 0,
        "class_weights": [1, 190123 / 66228],
        "allow_writing_files": False
    }

    model = CatBoostClassifier(**params)

    model.fit(
        X_train,
        y_train,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        early_stopping_rounds=80,
        verbose=0
    )

    pred = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, pred)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("best_score:", study.best_value)
print("best_params:", study.best_params)

[I 2026-05-25 18:50:11,981] A new study created in memory with name: no-name-3d9a0bbf-9c62-4db7-9fb2-9886b6f85db5
[I 2026-05-25 18:51:45,724] Trial 0 finished with value: 0.7373328417505469 and parameters: {'learning_rate': 0.018559325336749956, 'depth': 7, 'l2_leaf_reg': 1.6626100362675547, 'random_strength': 2.274549157385444, 'bagging_temperature': 1.8924243039840305}. Best is trial 0 with value: 0.7373328417505469.
[I 2026-05-25 18:52:43,790] Trial 1 finished with value: 0.7372059693159821 and parameters: {'learning_rate': 0.01803301862212673, 'depth': 7, 'l2_leaf_reg': 11.515375915736858, 'random_strength': 0.1844583448410941, 'bagging_temperature': 4.650943105432727}. Best is trial 0 with value: 0.7373328417505469.
[I 2026-05-25 18:53:21,264] Trial 2 finished with value: 0.7371855773660672 and parameters: {'learning_rate': 0.05281380149655695, 'depth': 7, 'l2_leaf_reg': 1.9962837926446988, 'random_strength': 1.9209448408411436, 'bagging_temperature': 2.6908173664601085}. Best is 

best_score: 0.737489457326157
best_params: {'learning_rate': 0.02498214961001344, 'depth': 8, 'l2_leaf_reg': 18.591182129683194, 'random_strength': 0.32969640414889206, 'bagging_temperature': 4.535604806522509}


## BEST PARMS

In [30]:
cat_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()


cat_model_2 = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.02498214961001344,
    depth=8,
    l2_leaf_reg=18.591182129683194,
    random_strength=0.32969640414889206,
    bagging_temperature=4.535604806522509,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    class_weights=[1, 190123 / 66228],
    allow_writing_files=False
)

cat_model_2.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50
)

y_val_pred = cat_model_2.predict(X_val)
y_val_proba = cat_model_2.predict_proba(X_val)[:, 1]

print("f1:", f1_score(y_val, y_val_pred))
print("precision:", precision_score(y_val, y_val_pred))
print("recall:", recall_score(y_val, y_val_pred))
print("roc_auc:", roc_auc_score(y_val, y_val_proba))

0:	test: 0.7211759	best: 0.7211759 (0)	total: 769ms	remaining: 25m 36s
100:	test: 0.7340689	best: 0.7340689 (100)	total: 41.6s	remaining: 13m 2s
200:	test: 0.7361859	best: 0.7361859 (200)	total: 1m 9s	remaining: 10m 23s
300:	test: 0.7368888	best: 0.7368888 (300)	total: 1m 25s	remaining: 8m 3s
400:	test: 0.7371866	best: 0.7371866 (400)	total: 1m 39s	remaining: 6m 37s
500:	test: 0.7372815	best: 0.7372831 (489)	total: 1m 53s	remaining: 5m 39s
600:	test: 0.7373504	best: 0.7373504 (600)	total: 2m 6s	remaining: 4m 53s
700:	test: 0.7373808	best: 0.7373901 (675)	total: 2m 18s	remaining: 4m 17s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7374159611
bestIteration = 739

Shrink model to first 740 iterations.
f1: 0.515556891627837
precision: 0.3857978404319136
recall: 0.7768382908047713
roc_auc: 0.7374159611331936


In [31]:
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": cat_model_2.get_feature_importance()
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

print(importance_df.head(30))

                feature  importance
84            배아_이식_집중도   27.605003
41             이식된 배아 수   21.134722
52                난자 출처    6.943069
1              시술 당시 나이    4.646812
80               배아_냉동률    3.999044
83   고령_난자수_interaction    3.938497
65            배아 이식 경과일    3.591906
70  배아 이식 경과일_performed    2.562485
47          수집된 신선 난자 수    2.561089
38            총 생성 배아 수    2.544344
79               배아_이식률    2.183246
43             저장된 배아 수    2.081833
3                 시술 유형    2.059326
53                정자 출처    1.610866
78               배아_생성률    0.889830
30            IVF 시술 횟수    0.834486
0              시술 시기 코드    0.815875
57          신선 배아 사용 여부    0.674580
81            IVF_임신성공률    0.663214
89            출산_임신_전환율    0.555973
77                 고령여부    0.536654
6              배란 유도 유형    0.489697
29        클리닉 내 총 시술 횟수    0.488551
73              남성_원인_수    0.383764
50     파트너 정자와 혼합된 난자 수    0.364233
55            정자 기증자 나이    0.358546
32              총 임신 횟수    0

## soft voting (cat, xgb, lgbm)

In [50]:
X_stack = X.copy()
y_stack = y.astype(int)

cat_cols = X_stack.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols:
    X_stack[col] = X_stack[col].astype(str)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_cat = np.zeros(len(X_stack))
oof_lgbm = np.zeros(len(X_stack))
oof_xgb = np.zeros(len(X_stack))

cat_scores = []
lgbm_scores = []
xgb_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
    print(f"\n===== Fold {fold} =====")

    X_train_fold = X_stack.iloc[train_idx].copy()
    X_val_fold = X_stack.iloc[val_idx].copy()
    y_train_fold = y_stack.iloc[train_idx]
    y_val_fold = y_stack.iloc[val_idx]

    # =====================
    # CatBoost: 원본 사용
    # =====================
    cat_model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.02498214961001344,
        depth=8,
        l2_leaf_reg=18.591182129683194,
        random_strength=0.32969640414889206,
        bagging_temperature=4.535604806522509,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=0,
        class_weights=[1, 190123 / 66228],
        allow_writing_files=False
    )

    cat_model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=cat_cols,
        eval_set=(X_val_fold, y_val_fold),
        early_stopping_rounds=100,
        verbose=0
    )

    cat_proba = cat_model.predict_proba(X_val_fold)[:, 1]
    oof_cat[val_idx] = cat_proba
    cat_auc = roc_auc_score(y_val_fold, cat_proba)
    cat_scores.append(cat_auc)

    # =====================
    # LGBM / XGB: preprocessor 사용
    # =====================
    X_train_trans = preprocessor.fit_transform(X_train_fold)
    X_val_trans = preprocessor.transform(X_val_fold)

    lgbm_model = LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )

    lgbm_model.fit(
        X_train_trans,
        y_train_fold,
        eval_set=[(X_val_trans, y_val_fold)],
        eval_metric="auc"
    )

    lgbm_proba = lgbm_model.predict_proba(X_val_trans)[:, 1]
    oof_lgbm[val_idx] = lgbm_proba
    lgbm_auc = roc_auc_score(y_val_fold, lgbm_proba)
    lgbm_scores.append(lgbm_auc)

    xgb_model = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=42,
        scale_pos_weight=190123 / 66228,
        tree_method="hist",
        n_jobs=-1
    )

    xgb_model.fit(
        X_train_trans,
        y_train_fold,
        eval_set=[(X_val_trans, y_val_fold)],
        verbose=False
    )

    xgb_proba = xgb_model.predict_proba(X_val_trans)[:, 1]
    oof_xgb[val_idx] = xgb_proba
    xgb_auc = roc_auc_score(y_val_fold, xgb_proba)
    xgb_scores.append(xgb_auc)

    print("Cat AUC :", cat_auc)
    print("LGBM AUC:", lgbm_auc)
    print("XGB AUC :", xgb_auc)


===== Fold 1 =====
[LightGBM] [Info] Number of positive: 52982, number of negative: 152098
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017811 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1594
[LightGBM] [Info] Number of data points in the train set: 205080, number of used features: 175
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Cat AUC : 0.7380011898447651
LGBM AUC: 0.7359912843324962
XGB AUC : 0.736638403634536

===== Fold 2 =====
[LightGBM] [Info] Number of positive: 52983, number of negative: 152098
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010794 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Inf

In [51]:
meta_X = np.column_stack([
    oof_cat,
    oof_lgbm,
    oof_xgb
])

meta_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

meta_model.fit(meta_X, y_stack)

stack_oof_proba = meta_model.predict_proba(meta_X)[:, 1]

print("\n====================")
print("Cat OOF AUC :", roc_auc_score(y_stack, oof_cat))
print("LGBM OOF AUC:", roc_auc_score(y_stack, oof_lgbm))
print("XGB OOF AUC :", roc_auc_score(y_stack, oof_xgb))
print("Stack OOF AUC:", roc_auc_score(y_stack, stack_oof_proba))

print("Meta coefficients:", meta_model.coef_)
print("Meta intercept:", meta_model.intercept_)


Cat OOF AUC : 0.7400729840303574
LGBM OOF AUC: 0.7378022697306811
XGB OOF AUC : 0.7387574799467093
Stack OOF AUC: 0.7401792252730631
Meta coefficients: [[ 4.23744943e+00 -3.65321227e-03  1.05717255e+00]]
Meta intercept: [-2.72287537]


## stacking(cat, xgb)

In [52]:
meta_X = np.column_stack([
    oof_cat,
    oof_xgb
])

meta_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

meta_model.fit(meta_X, y_stack)

stack_oof_proba = meta_model.predict_proba(meta_X)[:, 1]

print(
    "Cat+XGB Stack OOF AUC:",
    roc_auc_score(y_stack, stack_oof_proba)
)

print("Meta coefficients:", meta_model.coef_)

Cat+XGB Stack OOF AUC: 0.7401776650099504
Meta coefficients: [[4.35647081 0.93574586]]


## weight voting 0.8, 0.2 ㄱㄱ

In [53]:
for w in [0.6, 0.7, 0.8, 0.85, 0.9]:
    blend = w * oof_cat + (1 - w) * oof_xgb

    score = roc_auc_score(y_stack, blend)

    print(f"Cat {w:.2f} / XGB {1-w:.2f} -> {score:.6f}")

Cat 0.60 / XGB 0.40 -> 0.740083
Cat 0.70 / XGB 0.30 -> 0.740156
Cat 0.80 / XGB 0.20 -> 0.740179
Cat 0.85 / XGB 0.15 -> 0.740172
Cat 0.90 / XGB 0.10 -> 0.740152


## xgb tuning -> 별로

In [63]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y.astype(int),
    test_size=0.2,
    stratify=y.astype(int),
    random_state=42
)

y_train = y_train.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)

X_train_trans = preprocessor.fit_transform(X_train)
X_val_trans = preprocessor.transform(X_val)

print(X_train_trans.shape, len(y_train))
print(X_val_trans.shape, len(y_val))

def objective_xgb(trial):
    params = {
        "n_estimators": 800,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 20, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1, 30, log=True),
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "random_state": 42,
        "scale_pos_weight": 190123 / 66228,
        "tree_method": "hist",
        "n_jobs": -1,
    }

    model = XGBClassifier(**params)

    model.fit(
        X_train_trans,
        y_train.values,
        eval_set=[(X_val_trans, y_val.values)],
        verbose=False
    )

    pred = model.predict_proba(X_val_trans)[:, 1]

    return roc_auc_score(y_val, pred)

study_xgb = optuna.create_study(direction="maximize")

study_xgb.optimize(
    objective_xgb,
    n_trials=10
)

print(study_xgb.best_value)
print(study_xgb.best_params)

[I 2026-05-25 20:26:14,494] A new study created in memory with name: no-name-d25cb792-1dd9-45c2-8dab-9f45e3b8ae8c


(205080, 200) 205080
(51271, 200) 51271


[I 2026-05-25 20:26:29,445] Trial 0 finished with value: 0.7367105745790747 and parameters: {'learning_rate': 0.02531623132440725, 'max_depth': 4, 'min_child_weight': 1.4122695381430026, 'subsample': 0.6419369691905855, 'colsample_bytree': 0.9193195260438002, 'gamma': 0.07868635090847065, 'reg_alpha': 0.2508100579374867, 'reg_lambda': 3.9391233364134597}. Best is trial 0 with value: 0.7367105745790747.
[I 2026-05-25 20:26:35,647] Trial 1 finished with value: 0.7367363836680555 and parameters: {'learning_rate': 0.03824842521990857, 'max_depth': 8, 'min_child_weight': 13.11785858037057, 'subsample': 0.9903436449600395, 'colsample_bytree': 0.6082865363372915, 'gamma': 2.5212448828685403, 'reg_alpha': 6.0917350935034715, 'reg_lambda': 27.781750972688542}. Best is trial 1 with value: 0.7367363836680555.
[I 2026-05-25 20:26:42,386] Trial 2 finished with value: 0.7369775967498357 and parameters: {'learning_rate': 0.02197745933100438, 'max_depth': 4, 'min_child_weight': 13.148563786926157, 'su

0.7371435873412668
{'learning_rate': 0.02797308267328566, 'max_depth': 5, 'min_child_weight': 5.076998023944345, 'subsample': 0.6271417131586693, 'colsample_bytree': 0.7695007868111836, 'gamma': 4.497902002393211, 'reg_alpha': 0.1757137844134233, 'reg_lambda': 9.155608601426545}


## cat seed tuning

In [64]:
seeds = [42, 77, 2024, 2025, 3407]

cat_oof_seed_list = []

for seed in seeds:
    print(f"\n===== CatBoost seed {seed} =====")

    cat_model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.02498214961001344,
        depth=8,
        l2_leaf_reg=18.591182129683194,
        random_strength=0.32969640414889206,
        bagging_temperature=4.535604806522509,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=seed,
        verbose=100,
        class_weights=[1, 190123 / 66228],
        allow_writing_files=False
    )

    cat_model.fit(
        X_train,
        y_train,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100
    )

    cat_proba = cat_model.predict_proba(X_val)[:, 1]
    cat_oof_seed_list.append(cat_proba)

    print("AUC:", roc_auc_score(y_val, cat_proba))

cat_seed_proba = np.mean(cat_oof_seed_list, axis=0)

print("Cat seed ensemble AUC:", roc_auc_score(y_val, cat_seed_proba))


===== CatBoost seed 42 =====
0:	test: 0.7206621	best: 0.7206621 (0)	total: 2.28s	remaining: 1h 15m 50s
100:	test: 0.7339765	best: 0.7339765 (100)	total: 30.6s	remaining: 9m 36s
200:	test: 0.7362141	best: 0.7362141 (200)	total: 39.3s	remaining: 5m 51s
300:	test: 0.7368429	best: 0.7368472 (297)	total: 47.5s	remaining: 4m 27s
400:	test: 0.7370280	best: 0.7370322 (397)	total: 54.6s	remaining: 3m 37s
500:	test: 0.7372138	best: 0.7372138 (500)	total: 1m 2s	remaining: 3m 5s
600:	test: 0.7372766	best: 0.7372800 (594)	total: 1m 10s	remaining: 2m 44s
700:	test: 0.7373326	best: 0.7373326 (700)	total: 1m 19s	remaining: 2m 27s
800:	test: 0.7374438	best: 0.7374453 (797)	total: 1m 27s	remaining: 2m 11s
900:	test: 0.7374798	best: 0.7374798 (900)	total: 1m 36s	remaining: 1m 57s
1000:	test: 0.7375108	best: 0.7375108 (1000)	total: 1m 45s	remaining: 1m 44s
1100:	test: 0.7374649	best: 0.7375239 (1025)	total: 1m 53s	remaining: 1m 32s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7375

In [70]:
xgb_model.fit(
    X_train_trans,
    y_train.to_numpy().ravel(),
    eval_set=[(X_val_trans, y_val.to_numpy().ravel())],
    verbose=False
)

xgb_proba = xgb_model.predict_proba(X_val_trans)[:, 1]

print(len(cat_seed_proba), len(xgb_proba), len(y_val))

51271 51271 51271


## blend(cat seed, xgb)

In [71]:
for w in [0.75, 0.8, 0.85, 0.9]:
    blend = w * cat_seed_proba + (1 - w) * xgb_proba
    print(w, roc_auc_score(y_val, blend))

0.75 0.7375753969565744
0.8 0.7375919868829195
0.85 0.7375994062887058
0.9 0.7375904779858367


## TE

In [9]:
te_cols = [
    # 기존
    '시술 시기 코드',
    '시술 유형',
    '특정 시술 유형',
    '배란 유도 유형',
    '난자 출처',
    '정자 출처',
    '배아 생성 주요 이유',
    '시술 당시 나이',
]

te_cols = [col for col in te_cols if col in X.columns]

print(te_cols)
print(len(te_cols))

te_cols = [col for col in te_cols if col in X.columns]


def add_oof_target_encoding(X, y, cols, n_splits=5, smoothing=10):
    X = X.copy()
    y = y.astype(int).reset_index(drop=True)
    X = X.reset_index(drop=True)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    te_features = pd.DataFrame(index=X.index)

    for col in cols:
        te_features[f'{col}_TE'] = 0.0

    for train_idx, val_idx in skf.split(X, y):
        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_tr = y.iloc[train_idx]

        encoder = TargetEncoder(
            cols=cols,
            smoothing=smoothing
        )

        encoder.fit(X_tr[cols], y_tr)

        encoded_val = encoder.transform(X_val[cols])

        for col in cols:
            te_features.loc[val_idx, f'{col}_TE'] = encoded_val[col].values

    # 전체 train 기준 encoder도 같이 반환: test 변환용
    final_encoder = TargetEncoder(
        cols=cols,
        smoothing=smoothing
    )

    final_encoder.fit(X[cols], y)

    X_te = pd.concat([X, te_features], axis=1)

    return X_te, final_encoder

X_te, te_encoder = add_oof_target_encoding(
    X,
    y,
    cols=te_cols,
    n_splits=5,
    smoothing=10
)

print(X.shape)
print(X_te.shape)

['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '난자 출처', '정자 출처', '배아 생성 주요 이유', '시술 당시 나이']
8
(256351, 92)
(256351, 100)


TE V2

## oof ensemble
CatBoost 여러 seed OOF 평균 + XGB OOF blend

In [40]:
X_stack = X_te.copy()
y_stack = y.astype(int).reset_index(drop=True)

cat_cols = X_stack.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols:
    X_stack[col] = X_stack[col].astype(str)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cat_seeds = [42, 77, 2024]
cat_oof_seed_list = []
xgb_oof = np.zeros(len(X_stack))

for seed in cat_seeds:
    print(f"\n================ Cat seed {seed} ================")

    cat_oof = np.zeros(len(X_stack))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print(f"Cat Seed {seed} / Fold {fold}")

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        cat_model = CatBoostClassifier(
            iterations=2000,
            learning_rate=0.02498214961001344,
            depth=8,
            l2_leaf_reg=18.591182129683194,
            random_strength=0.32969640414889206,
            bagging_temperature=4.535604806522509,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=0,
            class_weights=[1, 190123 / 66228],
            allow_writing_files=False
        )

        cat_model.fit(
            X_tr,
            y_tr,
            cat_features=cat_cols,
            eval_set=(X_val, y_val),
            early_stopping_rounds=100,
            verbose=0
        )

        cat_oof[val_idx] = cat_model.predict_proba(X_val)[:, 1]

    print("Cat seed OOF AUC:", roc_auc_score(y_stack, cat_oof))
    cat_oof_seed_list.append(cat_oof)

cat_oof_seed = np.mean(cat_oof_seed_list, axis=0)

print("\nCat seed ensemble OOF AUC:", roc_auc_score(y_stack, cat_oof_seed))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
    print(f"\nXGB Fold {fold}")

    X_tr = X_stack.iloc[tr_idx].copy()
    X_val = X_stack.iloc[val_idx].copy()
    y_tr = y_stack.iloc[tr_idx]
    y_val = y_stack.iloc[val_idx]

    X_tr_trans = preprocessor.fit_transform(X_tr)
    X_val_trans = preprocessor.transform(X_val)

    xgb_model = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=42,
        scale_pos_weight=190123 / 66228,
        tree_method="hist",
        n_jobs=-1
    )

    xgb_model.fit(
        X_tr_trans,
        y_tr.to_numpy().ravel(),
        eval_set=[(X_val_trans, y_val.to_numpy().ravel())],
        verbose=False
    )

    xgb_oof[val_idx] = xgb_model.predict_proba(X_val_trans)[:, 1]

print("XGB OOF AUC:", roc_auc_score(y_stack, xgb_oof))

for w in [0.75, 0.8, 0.85, 0.9, 0.95]:
    blend = w * cat_oof_seed + (1 - w) * xgb_oof
    print(w, roc_auc_score(y_stack, blend))


================ Cat seed 42 ================
Cat Seed 42 / Fold 1


KeyboardInterrupt: 

In [93]:
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

cat_rank = rankdata(cat_oof_seed) / len(cat_oof_seed)
xgb_rank = rankdata(xgb_oof) / len(xgb_oof)

for w in [0.75, 0.8, 0.85, 0.9]:
    blend_rank = w * cat_rank + (1 - w) * xgb_rank
    print(w, roc_auc_score(y_stack, blend_rank))

0.75 0.7402447114124147
0.8 0.740252106778425
0.85 0.7402477512887776
0.9 0.7402328587020569


## TE smoothing

In [85]:
smoothing_list = [3, 5, 10, 20, 50]

smoothing_scores = {}

for smoothing in smoothing_list:
    print(f"\n================ smoothing={smoothing} ================")

    X_te, te_encoder = add_oof_target_encoding(
        X,
        y,
        cols=te_cols,
        n_splits=5,
        smoothing=smoothing
    )

    X_stack = X_te.copy()
    y_stack = y.astype(int).reset_index(drop=True)

    cat_cols = X_stack.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        X_stack[col] = X_stack[col].astype(str)

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    cat_oof = np.zeros(len(X_stack))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print(f"Fold {fold}")

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        cat_model = CatBoostClassifier(
            iterations=2000,
            learning_rate=0.02498214961001344,
            depth=8,
            l2_leaf_reg=18.591182129683194,
            random_strength=0.32969640414889206,
            bagging_temperature=4.535604806522509,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42,
            verbose=0,
            class_weights=[1, 190123 / 66228],
            allow_writing_files=False
        )

        cat_model.fit(
            X_tr,
            y_tr,
            cat_features=cat_cols,
            eval_set=(X_val, y_val),
            early_stopping_rounds=100,
            verbose=0
        )

        cat_oof[val_idx] = cat_model.predict_proba(X_val)[:, 1]

    score = roc_auc_score(y_stack, cat_oof)
    smoothing_scores[smoothing] = score

    print(f"smoothing={smoothing}, Cat OOF AUC={score:.6f}")

print("\n===== 결과 =====")
for smoothing, score in smoothing_scores.items():
    print(smoothing, score)


================ smoothing=3 ================
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
smoothing=3, Cat OOF AUC=0.739980

================ smoothing=5 ================
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
smoothing=5, Cat OOF AUC=0.740163

================ smoothing=10 ================
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
smoothing=10, Cat OOF AUC=0.740051

================ smoothing=20 ================
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
smoothing=20, Cat OOF AUC=0.739973

================ smoothing=50 ================
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
smoothing=50, Cat OOF AUC=0.740111

===== 결과 =====
3 0.7399801462308577
5 0.7401631784919105
10 0.7400514463476879
20 0.7399729432173499
50 0.7401112976388216


## TE COMBO 생성

In [10]:
def add_combo_columns(X):
    X = X.copy()

    combo_pairs = [
        ("시술 당시 나이", "난자 출처"),
        ("시술 당시 나이", "정자 출처"),
        ("시술 당시 나이", "시술 유형"),
        ("시술 당시 나이", "특정 시술 유형"),
        ("시술 유형", "난자 출처"),
        ("시술 유형", "정자 출처"),
        ("특정 시술 유형", "난자 출처"),
        ("특정 시술 유형", "정자 출처"),
        ("난자 출처", "정자 출처"),
        ("배란 유도 유형", "시술 당시 나이"),
    ]

    combo_cols = []

    for col1, col2 in combo_pairs:
        if col1 in X.columns and col2 in X.columns:
            new_col = f"{col1}_{col2}_combo"
            X[new_col] = (
                X[col1].astype(str) + "_" + X[col2].astype(str)
            )
            combo_cols.append(new_col)

    return X, combo_cols

In [11]:
X_combo, combo_cols = add_combo_columns(X)

print(combo_cols)
print(len(combo_cols))

base_te_cols = [
    '시술 시기 코드',
    '시술 유형',
    '특정 시술 유형',
    '배란 유도 유형',
    '난자 출처',
    '정자 출처',
    '배아 생성 주요 이유',
    '시술 당시 나이',
]

base_te_cols = [col for col in base_te_cols if col in X_combo.columns]

te_cols = base_te_cols + combo_cols

print(te_cols)
print(len(te_cols))

X_te_combo, te_encoder_combo = add_oof_target_encoding(
    X_combo,
    y,
    cols=te_cols,
    n_splits=5,
    smoothing=10
)

print(X_combo.shape)
print(X_te_combo.shape)

X_te_combo = X_te_combo.drop(columns=combo_cols, errors="ignore")

print(X_te_combo.shape)

['시술 당시 나이_난자 출처_combo', '시술 당시 나이_정자 출처_combo', '시술 당시 나이_시술 유형_combo', '시술 당시 나이_특정 시술 유형_combo', '시술 유형_난자 출처_combo', '시술 유형_정자 출처_combo', '특정 시술 유형_난자 출처_combo', '특정 시술 유형_정자 출처_combo', '난자 출처_정자 출처_combo', '배란 유도 유형_시술 당시 나이_combo']
10
['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '난자 출처', '정자 출처', '배아 생성 주요 이유', '시술 당시 나이', '시술 당시 나이_난자 출처_combo', '시술 당시 나이_정자 출처_combo', '시술 당시 나이_시술 유형_combo', '시술 당시 나이_특정 시술 유형_combo', '시술 유형_난자 출처_combo', '시술 유형_정자 출처_combo', '특정 시술 유형_난자 출처_combo', '특정 시술 유형_정자 출처_combo', '난자 출처_정자 출처_combo', '배란 유도 유형_시술 당시 나이_combo']
18
(256351, 102)
(256351, 120)
(256351, 110)


## Adversarial Validation

In [94]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

# train/test feature는 같은 preprocessing 적용된 상태여야 함
# X: train feature
# X_test: test feature 또는 test_processed

X_adv_train = X.copy()
X_adv_test = X_test.copy()   # 변수명 맞게 수정: test_processed면 그걸 넣기

X_adv_train["is_test"] = 0
X_adv_test["is_test"] = 1

adv_df = pd.concat(
    [X_adv_train, X_adv_test],
    axis=0
).reset_index(drop=True)

y_adv = adv_df["is_test"].astype(int)
X_adv = adv_df.drop(columns=["is_test"])

cat_cols_adv = X_adv.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols_adv:
    X_adv[col] = X_adv[col].astype(str)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

adv_oof = np.zeros(len(X_adv))
adv_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_adv, y_adv), 1):
    print(f"\n===== Adversarial Fold {fold} =====")

    X_tr = X_adv.iloc[tr_idx].copy()
    X_val = X_adv.iloc[val_idx].copy()
    y_tr = y_adv.iloc[tr_idx]
    y_val = y_adv.iloc[val_idx]

    adv_model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=100,
        allow_writing_files=False
    )

    adv_model.fit(
        X_tr,
        y_tr,
        cat_features=cat_cols_adv,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=100
    )

    val_pred = adv_model.predict_proba(X_val)[:, 1]
    adv_oof[val_idx] = val_pred

    auc = roc_auc_score(y_val, val_pred)
    adv_scores.append(auc)

    print(f"Fold {fold} Adv AUC: {auc:.6f}")

print("\n====================")
print("Adversarial Fold AUCs:", adv_scores)
print("Adversarial Mean AUC:", np.mean(adv_scores))
print("Adversarial OOF AUC:", roc_auc_score(y_adv, adv_oof))


===== Adversarial Fold 1 =====
0:	test: 0.5048369	best: 0.5048369 (0)	total: 113ms	remaining: 1m 52s
100:	test: 0.4976078	best: 0.5048369 (0)	total: 27.1s	remaining: 4m
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.5048368903
bestIteration = 0

Shrink model to first 1 iterations.
Fold 1 Adv AUC: 0.504837

===== Adversarial Fold 2 =====
0:	test: 0.4990010	best: 0.4990010 (0)	total: 110ms	remaining: 1m 49s
100:	test: 0.4988424	best: 0.5022294 (1)	total: 8.57s	remaining: 1m 16s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.5022294246
bestIteration = 1

Shrink model to first 2 iterations.
Fold 2 Adv AUC: 0.502229

===== Adversarial Fold 3 =====
0:	test: 0.5003110	best: 0.5003110 (0)	total: 86.3ms	remaining: 1m 26s
100:	test: 0.5000510	best: 0.5003788 (59)	total: 8.28s	remaining: 1m 13s
200:	test: 0.4981632	best: 0.5009305 (109)	total: 17.3s	remaining: 1m 8s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.500930476
bestIter

## shallow cat oof

In [116]:
cat_shallow_oof = np.zeros(len(X_stack))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
    print(f"\n===== Shallow Cat Fold {fold} =====")

    X_tr = X_stack.iloc[tr_idx].copy()
    X_val = X_stack.iloc[val_idx].copy()
    y_tr = y_stack.iloc[tr_idx]
    y_val = y_stack.iloc[val_idx]

    cat_shallow = CatBoostClassifier(
        iterations=2500,
        learning_rate=0.02,
        depth=5,
        l2_leaf_reg=8,
        random_strength=1.5,
        bagging_temperature=2,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=100,
        class_weights=[1, 190123 / 66228],
        allow_writing_files=False
    )

    cat_shallow.fit(
        X_tr,
        y_tr,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        early_stopping_rounds=150,
        verbose=100
    )

    cat_shallow_oof[val_idx] = cat_shallow.predict_proba(X_val)[:, 1]

print("Shallow Cat OOF AUC:", roc_auc_score(y_stack, cat_shallow_oof))


===== Shallow Cat Fold 1 =====
0:	test: 0.7105843	best: 0.7105843 (0)	total: 118ms	remaining: 4m 55s
100:	test: 0.7314053	best: 0.7314053 (100)	total: 5.38s	remaining: 2m 7s
200:	test: 0.7341033	best: 0.7341033 (200)	total: 10.2s	remaining: 1m 56s
300:	test: 0.7354685	best: 0.7354685 (300)	total: 15.1s	remaining: 1m 50s
400:	test: 0.7362301	best: 0.7362301 (400)	total: 20.2s	remaining: 1m 45s
500:	test: 0.7367547	best: 0.7367547 (500)	total: 25.2s	remaining: 1m 40s
600:	test: 0.7370959	best: 0.7370959 (600)	total: 30.1s	remaining: 1m 35s
700:	test: 0.7375917	best: 0.7375917 (700)	total: 35s	remaining: 1m 29s
800:	test: 0.7378810	best: 0.7378810 (800)	total: 40.1s	remaining: 1m 25s
900:	test: 0.7380160	best: 0.7380171 (898)	total: 45.2s	remaining: 1m 20s
1000:	test: 0.7380332	best: 0.7380502 (954)	total: 50.6s	remaining: 1m 15s
1100:	test: 0.7380567	best: 0.7380582 (1097)	total: 56.5s	remaining: 1m 11s
1200:	test: 0.7380721	best: 0.7380989 (1148)	total: 1m 1s	remaining: 1m 6s
Stopped b

In [97]:
from scipy.stats import rankdata

print("Probability blend")
for w_main in [0.7, 0.75, 0.8]:
    for w_shallow in [0.05, 0.1, 0.15]:
        w_xgb = 1 - w_main - w_shallow

        if w_xgb <= 0:
            continue

        blend = (
            w_main * cat_oof_seed +
            w_shallow * cat_shallow_oof +
            w_xgb * xgb_oof
        )

        print(
            f"MainCat {w_main:.2f} / ShallowCat {w_shallow:.2f} / XGB {w_xgb:.2f}:",
            roc_auc_score(y_stack, blend)
        )

print("\nRank blend")
main_cat_rank = rankdata(cat_oof_seed) / len(cat_oof_seed)
shallow_cat_rank = rankdata(cat_shallow_oof) / len(cat_shallow_oof)
xgb_rank = rankdata(xgb_oof) / len(xgb_oof)

for w_main in [0.7, 0.75, 0.8]:
    for w_shallow in [0.05, 0.1, 0.15]:
        w_xgb = 1 - w_main - w_shallow

        if w_xgb <= 0:
            continue

        blend_rank = (
            w_main * main_cat_rank +
            w_shallow * shallow_cat_rank +
            w_xgb * xgb_rank
        )

        print(
            f"Rank MainCat {w_main:.2f} / ShallowCat {w_shallow:.2f} / XGB {w_xgb:.2f}:",
            roc_auc_score(y_stack, blend_rank)
        )

Probability blend
MainCat 0.70 / ShallowCat 0.05 / XGB 0.25: 0.7402389295598691
MainCat 0.70 / ShallowCat 0.10 / XGB 0.20: 0.7402643766363954
MainCat 0.70 / ShallowCat 0.15 / XGB 0.15: 0.7402772872855152
MainCat 0.75 / ShallowCat 0.05 / XGB 0.20: 0.7402530824789476
MainCat 0.75 / ShallowCat 0.10 / XGB 0.15: 0.740267158599979
MainCat 0.75 / ShallowCat 0.15 / XGB 0.10: 0.7402685479536841
MainCat 0.80 / ShallowCat 0.05 / XGB 0.15: 0.7402545070152118
MainCat 0.80 / ShallowCat 0.10 / XGB 0.10: 0.7402574335211383
MainCat 0.80 / ShallowCat 0.15 / XGB 0.05: 0.7402480529613539

Rank blend
Rank MainCat 0.70 / ShallowCat 0.05 / XGB 0.25: 0.7402581473379384
Rank MainCat 0.70 / ShallowCat 0.10 / XGB 0.20: 0.7402787463292785
Rank MainCat 0.70 / ShallowCat 0.15 / XGB 0.15: 0.7402874491284297
Rank MainCat 0.75 / ShallowCat 0.05 / XGB 0.20: 0.7402670885525363
Rank MainCat 0.75 / ShallowCat 0.10 / XGB 0.15: 0.7402768713688952
Rank MainCat 0.75 / ShallowCat 0.15 / XGB 0.10: 0.7402752821178954
Rank MainCa

In [98]:
for w_main in [0.65, 0.68, 0.70, 0.72, 0.75]:
    for w_shallow in [0.10, 0.12, 0.15, 0.18, 0.20]:
        w_xgb = 1 - w_main - w_shallow
        if w_xgb <= 0:
            continue

        blend_rank = (
            w_main * main_cat_rank +
            w_shallow * shallow_cat_rank +
            w_xgb * xgb_rank
        )

        print(
            f"Rank MainCat {w_main:.2f} / ShallowCat {w_shallow:.2f} / XGB {w_xgb:.2f}:",
            roc_auc_score(y_stack, blend_rank)
        )

Rank MainCat 0.65 / ShallowCat 0.10 / XGB 0.25: 0.7402690990015109
Rank MainCat 0.65 / ShallowCat 0.12 / XGB 0.23: 0.7402780740485473
Rank MainCat 0.65 / ShallowCat 0.15 / XGB 0.20: 0.7402879713074975
Rank MainCat 0.65 / ShallowCat 0.18 / XGB 0.17: 0.740293650233188
Rank MainCat 0.65 / ShallowCat 0.20 / XGB 0.15: 0.7402950781050448
Rank MainCat 0.68 / ShallowCat 0.10 / XGB 0.22: 0.7402763001486754
Rank MainCat 0.68 / ShallowCat 0.12 / XGB 0.20: 0.7402828738073512
Rank MainCat 0.68 / ShallowCat 0.15 / XGB 0.17: 0.7402889598341675
Rank MainCat 0.68 / ShallowCat 0.18 / XGB 0.14: 0.740290982950452
Rank MainCat 0.68 / ShallowCat 0.20 / XGB 0.12: 0.7402897478281918
Rank MainCat 0.70 / ShallowCat 0.10 / XGB 0.20: 0.7402787463292785
Rank MainCat 0.70 / ShallowCat 0.12 / XGB 0.18: 0.7402838657490327
Rank MainCat 0.70 / ShallowCat 0.15 / XGB 0.15: 0.7402874491284297
Rank MainCat 0.70 / ShallowCat 0.18 / XGB 0.12: 0.7402868636128135
Rank MainCat 0.70 / ShallowCat 0.20 / XGB 0.10: 0.74028356089970

## another cat(3) 추가

In [117]:
cat_random_oof = np.zeros(len(X_stack))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
    print(f"\n===== Random Cat Fold {fold} =====")

    X_tr = X_stack.iloc[tr_idx].copy()
    X_val = X_stack.iloc[val_idx].copy()
    y_tr = y_stack.iloc[tr_idx]
    y_val = y_stack.iloc[val_idx]

    cat_random = CatBoostClassifier(
        iterations=2500,
        learning_rate=0.022,
        depth=7,
        l2_leaf_reg=15,
        random_strength=8,
        bagging_temperature=8,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=100,
        class_weights=[1, 190123 / 66228],
        allow_writing_files=False
    )

    cat_random.fit(
        X_tr,
        y_tr,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        early_stopping_rounds=200,
        verbose=100
    )

    cat_random_oof[val_idx] = cat_random.predict_proba(X_val)[:, 1]

print("Random Cat OOF AUC:", roc_auc_score(y_stack, cat_random_oof))


===== Random Cat Fold 1 =====
0:	test: 0.6769145	best: 0.6769145 (0)	total: 145ms	remaining: 6m 1s
100:	test: 0.7285314	best: 0.7285314 (100)	total: 7.29s	remaining: 2m 53s
200:	test: 0.7312200	best: 0.7312200 (200)	total: 13s	remaining: 2m 29s
300:	test: 0.7326119	best: 0.7326160 (299)	total: 19.5s	remaining: 2m 22s
400:	test: 0.7337741	best: 0.7337741 (399)	total: 25.9s	remaining: 2m 15s
500:	test: 0.7342814	best: 0.7342814 (500)	total: 31.5s	remaining: 2m 5s
600:	test: 0.7351143	best: 0.7351143 (600)	total: 37.3s	remaining: 1m 57s
700:	test: 0.7368959	best: 0.7368959 (700)	total: 44.1s	remaining: 1m 53s
800:	test: 0.7376979	best: 0.7376979 (800)	total: 51s	remaining: 1m 48s
900:	test: 0.7378905	best: 0.7378982 (899)	total: 58s	remaining: 1m 42s
1000:	test: 0.7379849	best: 0.7379996 (983)	total: 1m 4s	remaining: 1m 37s
1100:	test: 0.7380117	best: 0.7380330 (1094)	total: 1m 11s	remaining: 1m 31s
1200:	test: 0.7380114	best: 0.7380589 (1149)	total: 1m 18s	remaining: 1m 25s
1300:	test: 

In [104]:
from scipy.stats import rankdata

main_cat_rank = rankdata(cat_oof_seed) / len(cat_oof_seed)
shallow_cat_rank = rankdata(cat_shallow_oof) / len(cat_shallow_oof)
random_cat_rank = rankdata(cat_random_oof) / len(cat_random_oof)
xgb_rank = rankdata(xgb_oof) / len(xgb_oof)

best_score = 0
best_weights = None

for w_main in np.arange(0.40, 0.61, 0.02):
    for w_shallow in np.arange(0.20, 0.41, 0.02):
        for w_random in np.arange(0.00, 0.21, 0.02):
            w_xgb = 1 - w_main - w_shallow - w_random

            if w_xgb <= 0:
                continue

            blend_rank = (
                w_main * main_cat_rank +
                w_shallow * shallow_cat_rank +
                w_random * random_cat_rank +
                w_xgb * xgb_rank
            )

            score = roc_auc_score(y_stack, blend_rank)

            if score > best_score:
                best_score = score
                best_weights = (w_main, w_shallow, w_random, w_xgb)

print("best_score:", best_score)
print("best_weights:", best_weights)

best_score: 0.7403297876454966
best_weights: (np.float64(0.44000000000000006), np.float64(0.2), np.float64(0.2), np.float64(0.15999999999999992))


In [41]:
np.save("../oof_preds/base_te_s10/cat_oof_seed.npy", cat_oof_seed)
np.save("../oof_preds/base_te_s10/xgb_oof.npy", xgb_oof)
np.save("../oof_preds/base_te_s10/cat_shallow_oof.npy", cat_shallow_oof)
np.save("../oof_preds/base_te_s10/cat_random_oof.npy", cat_random_oof)
np.save("../oof_preds/base_te_s10/y_stack.npy", y_stack.to_numpy())

NameError: name 'cat_oof_seed' is not defined

In [42]:
cat_oof_seed = np.load("../oof_preds/base_te_s10/cat_oof_seed.npy")
xgb_oof = np.load("../oof_preds/base_te_s10/xgb_oof.npy")
cat_shallow_oof = np.load("../oof_preds/base_te_s10/cat_shallow_oof.npy")
cat_random_oof = np.load("../oof_preds/base_te_s10/cat_random_oof.npy")
y_stack = np.load("../oof_preds/base_te_s10/y_stack.npy")

## Combo TE용 작업 변수 세팅

In [42]:
X_stack = X_te_combo.copy()
y_stack = y.astype(int).reset_index(drop=True)

cat_cols = X_stack.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols:
    X_stack[col] = X_stack[col].astype(str)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## Combo TE 기준 Main Cat seed ensemble 생성

In [52]:
cat_seeds = [42, 77, 2024]

combo_cat_oof_seed_list = []

for seed in cat_seeds:
    print(f"\n================ Combo Main Cat seed {seed} ================")

    cat_oof = np.zeros(len(X_stack))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print(f"Combo Main Cat Seed {seed} / Fold {fold}")

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        cat_model = CatBoostClassifier(
            iterations=2000,
            learning_rate=0.02498214961001344,
            depth=8,
            l2_leaf_reg=18.591182129683194,
            random_strength=0.32969640414889206,
            bagging_temperature=4.535604806522509,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=0,
            class_weights=[1, 190123 / 66228],
            allow_writing_files=False
        )

        cat_model.fit(
            X_tr,
            y_tr,
            cat_features=cat_cols,
            eval_set=(X_val, y_val),
            early_stopping_rounds=100,
            verbose=0
        )

        cat_oof[val_idx] = cat_model.predict_proba(X_val)[:, 1]

    print("Combo Main Cat seed OOF AUC:", roc_auc_score(y_stack, cat_oof))
    combo_cat_oof_seed_list.append(cat_oof)

combo_cat_oof_seed = np.mean(combo_cat_oof_seed_list, axis=0)

print(
    "Combo Main Cat seed ensemble OOF AUC:",
    roc_auc_score(y_stack, combo_cat_oof_seed)
)

np.save(
    "../oof_preds/combo_te_s10/combo_cat_oof_seed.npy",
    combo_cat_oof_seed
)


================ Combo Main Cat seed 42 ================
Combo Main Cat Seed 42 / Fold 1
Combo Main Cat Seed 42 / Fold 2
Combo Main Cat Seed 42 / Fold 3
Combo Main Cat Seed 42 / Fold 4
Combo Main Cat Seed 42 / Fold 5
Combo Main Cat seed OOF AUC: 0.7401624698373367

================ Combo Main Cat seed 77 ================
Combo Main Cat Seed 77 / Fold 1
Combo Main Cat Seed 77 / Fold 2
Combo Main Cat Seed 77 / Fold 3
Combo Main Cat Seed 77 / Fold 4
Combo Main Cat Seed 77 / Fold 5
Combo Main Cat seed OOF AUC: 0.7402104992723436

================ Combo Main Cat seed 2024 ================
Combo Main Cat Seed 2024 / Fold 1
Combo Main Cat Seed 2024 / Fold 2
Combo Main Cat Seed 2024 / Fold 3
Combo Main Cat Seed 2024 / Fold 4
Combo Main Cat Seed 2024 / Fold 5
Combo Main Cat seed OOF AUC: 0.7402534998648169
Combo Main Cat seed ensemble OOF AUC: 0.7403475273589835


In [53]:
combo_xgb_oof = np.zeros(len(X_stack))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
    print(f"\nCombo XGB Fold {fold}")

    X_tr = X_stack.iloc[tr_idx].copy()
    X_val = X_stack.iloc[val_idx].copy()
    y_tr = y_stack.iloc[tr_idx]
    y_val = y_stack.iloc[val_idx]

    X_tr_trans = preprocessor.fit_transform(X_tr)
    X_val_trans = preprocessor.transform(X_val)

    xgb_model = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=42,
        scale_pos_weight=190123 / 66228,
        tree_method="hist",
        n_jobs=-1
    )

    xgb_model.fit(
        X_tr_trans,
        y_tr.to_numpy().ravel(),
        eval_set=[(X_val_trans, y_val.to_numpy().ravel())],
        verbose=False
    )

    combo_xgb_oof[val_idx] = xgb_model.predict_proba(X_val_trans)[:, 1]

print("Combo XGB OOF AUC:", roc_auc_score(y_stack, combo_xgb_oof))

np.save(
    "../oof_preds/combo_te_s10/combo_xgb_oof.npy",
    combo_xgb_oof
)


Combo XGB Fold 1

Combo XGB Fold 2

Combo XGB Fold 3

Combo XGB Fold 4

Combo XGB Fold 5
Combo XGB OOF AUC: 0.7387325766114737


In [54]:
combo_cat_shallow_oof = np.zeros(len(X_stack))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
    print(f"\nCombo Shallow Cat Fold {fold}")

    X_tr = X_stack.iloc[tr_idx].copy()
    X_val = X_stack.iloc[val_idx].copy()
    y_tr = y_stack.iloc[tr_idx]
    y_val = y_stack.iloc[val_idx]

    cat_shallow = CatBoostClassifier(
        iterations=2500,
        learning_rate=0.02,
        depth=5,
        l2_leaf_reg=8,
        random_strength=1.5,
        bagging_temperature=2,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=0,
        class_weights=[1, 190123 / 66228],
        allow_writing_files=False
    )

    cat_shallow.fit(
        X_tr,
        y_tr,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        early_stopping_rounds=150,
        verbose=0
    )

    combo_cat_shallow_oof[val_idx] = cat_shallow.predict_proba(X_val)[:, 1]

print(
    "Combo Shallow Cat OOF AUC:",
    roc_auc_score(y_stack, combo_cat_shallow_oof)
)

np.save(
    "../oof_preds/combo_te_s10/combo_cat_shallow_oof.npy",
    combo_cat_shallow_oof
)


Combo Shallow Cat Fold 1

Combo Shallow Cat Fold 2

Combo Shallow Cat Fold 3

Combo Shallow Cat Fold 4

Combo Shallow Cat Fold 5
Combo Shallow Cat OOF AUC: 0.7401183538862586


In [55]:
combo_cat_random_oof = np.zeros(len(X_stack))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
    print(f"\nCombo Random Cat Fold {fold}")

    X_tr = X_stack.iloc[tr_idx].copy()
    X_val = X_stack.iloc[val_idx].copy()
    y_tr = y_stack.iloc[tr_idx]
    y_val = y_stack.iloc[val_idx]

    cat_random = CatBoostClassifier(
        iterations=2500,
        learning_rate=0.022,
        depth=7,
        l2_leaf_reg=15,
        random_strength=8,
        bagging_temperature=8,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=0,
        class_weights=[1, 190123 / 66228],
        allow_writing_files=False
    )

    cat_random.fit(
        X_tr,
        y_tr,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        early_stopping_rounds=200,
        verbose=0
    )

    combo_cat_random_oof[val_idx] = cat_random.predict_proba(X_val)[:, 1]

print(
    "Combo Random Cat OOF AUC:",
    roc_auc_score(y_stack, combo_cat_random_oof)
)

np.save(
    "../oof_preds/combo_te_s10/combo_cat_random_oof.npy",
    combo_cat_random_oof
)


Combo Random Cat Fold 1

Combo Random Cat Fold 2

Combo Random Cat Fold 3

Combo Random Cat Fold 4

Combo Random Cat Fold 5
Combo Random Cat OOF AUC: 0.7402861182667221


In [56]:
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

combo_main_cat_rank = rankdata(combo_cat_oof_seed) / len(combo_cat_oof_seed)
combo_shallow_cat_rank = rankdata(combo_cat_shallow_oof) / len(combo_cat_shallow_oof)
combo_random_cat_rank = rankdata(combo_cat_random_oof) / len(combo_cat_random_oof)
combo_xgb_rank = rankdata(combo_xgb_oof) / len(combo_xgb_oof)

best_score = 0
best_weights = None

for w_main in np.arange(0.25, 0.51, 0.01):
    for w_shallow in np.arange(0.08, 0.25, 0.01):
        for w_random in np.arange(0.25, 0.51, 0.01):
            w_xgb = 1 - w_main - w_shallow - w_random

            if w_xgb <= 0:
                continue

            blend_rank = (
                w_main * combo_main_cat_rank +
                w_shallow * combo_shallow_cat_rank +
                w_random * combo_random_cat_rank +
                w_xgb * combo_xgb_rank
            )

            score = roc_auc_score(y_stack, blend_rank)

            if score > best_score:
                best_score = score
                best_weights = (w_main, w_shallow, w_random, w_xgb)

print("Combo TE best_score:", best_score)
print("Combo TE best_weights:", best_weights)

Combo TE best_score: 0.7404963281414699
Combo TE best_weights: (np.float64(0.44000000000000017), np.float64(0.08), np.float64(0.3500000000000001), np.float64(0.12999999999999973))


In [57]:
np.save("../oof_preds/combo_te_s10/combo_main_cat_rank.npy", combo_main_cat_rank)
np.save("../oof_preds/combo_te_s10/combo_shallow_cat_rank.npy", combo_shallow_cat_rank)
np.save("../oof_preds/combo_te_s10/combo_random_cat_rank.npy", combo_random_cat_rank)
np.save("../oof_preds/combo_te_s10/combo_xgb_rank.npy", combo_xgb_rank)

final_combo_rank = (
    0.44 * combo_main_cat_rank +
    0.08 * combo_shallow_cat_rank +
    0.35 * combo_random_cat_rank +
    0.13 * combo_xgb_rank
)

np.save("../oof_preds/combo_te_s10/final_combo_rank.npy", final_combo_rank)


## cat random 2 실험

In [43]:
import numpy as np
from pathlib import Path
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

save_dir = Path("../oof_preds/combo_te_s10")

combo_cat_oof_seed = np.load(save_dir / "combo_cat_oof_seed.npy")
combo_xgb_oof = np.load(save_dir / "combo_xgb_oof.npy")
combo_cat_shallow_oof = np.load(save_dir / "combo_cat_shallow_oof.npy")
combo_cat_random_oof = np.load(save_dir / "combo_cat_random_oof.npy")
y_stack = np.load("../oof_preds/base_te_s10/y_stack.npy")

combo_main_cat_rank = rankdata(combo_cat_oof_seed) / len(combo_cat_oof_seed)
combo_shallow_cat_rank = rankdata(combo_cat_shallow_oof) / len(combo_cat_shallow_oof)
combo_random_cat_rank = rankdata(combo_cat_random_oof) / len(combo_cat_random_oof)
combo_xgb_rank = rankdata(combo_xgb_oof) / len(combo_xgb_oof)

final_combo_rank = (
    0.44 * combo_main_cat_rank +
    0.08 * combo_shallow_cat_rank +
    0.35 * combo_random_cat_rank +
    0.13 * combo_xgb_rank
)

roc_auc_score(y_stack, final_combo_rank)

0.7404963277840849

In [45]:
y_stack = pd.Series(y_stack).reset_index(drop=True)
combo_cat_random2_oof = np.zeros(len(X_stack))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
    print(f"\nCombo Random Cat 2 Fold {fold}")

    X_tr = X_stack.iloc[tr_idx].copy()
    X_val = X_stack.iloc[val_idx].copy()
    y_tr = y_stack.iloc[tr_idx]
    y_val = y_stack.iloc[val_idx]

    cat_random2 = CatBoostClassifier(
        iterations=2500,
        learning_rate=0.018,
        depth=7,
        l2_leaf_reg=25,
        random_strength=12,
        bagging_temperature=10,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=2024,
        verbose=100,
        class_weights=[1, 190123 / 66228],
        allow_writing_files=False
    )

    cat_random2.fit(
        X_tr,
        y_tr,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        early_stopping_rounds=200,
        verbose=100
    )

    combo_cat_random2_oof[val_idx] = cat_random2.predict_proba(X_val)[:, 1]

print(
    "Combo Random Cat 2 OOF AUC:",
    roc_auc_score(y_stack, combo_cat_random2_oof)
)


Combo Random Cat 2 Fold 1
0:	test: 0.6989218	best: 0.6989218 (0)	total: 3.1s	remaining: 2h 9m 12s
100:	test: 0.7254072	best: 0.7254830 (96)	total: 36s	remaining: 14m 14s
200:	test: 0.7283302	best: 0.7283312 (199)	total: 58.3s	remaining: 11m 6s
300:	test: 0.7301919	best: 0.7301919 (300)	total: 1m 25s	remaining: 10m 23s
400:	test: 0.7312810	best: 0.7312810 (400)	total: 1m 58s	remaining: 10m 22s
500:	test: 0.7321668	best: 0.7321668 (500)	total: 2m 8s	remaining: 8m 33s
600:	test: 0.7329152	best: 0.7329176 (595)	total: 2m 18s	remaining: 7m 17s
700:	test: 0.7333899	best: 0.7333899 (700)	total: 2m 27s	remaining: 6m 19s
800:	test: 0.7350965	best: 0.7350965 (800)	total: 2m 39s	remaining: 5m 37s
900:	test: 0.7368836	best: 0.7368836 (900)	total: 2m 51s	remaining: 5m 3s
1000:	test: 0.7376087	best: 0.7376087 (1000)	total: 3m 2s	remaining: 4m 33s
1100:	test: 0.7378945	best: 0.7378945 (1100)	total: 3m 14s	remaining: 4m 7s
1200:	test: 0.7380096	best: 0.7380096 (1200)	total: 3m 25s	remaining: 3m 42s
1

In [52]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import numpy as np

# =========================
# CatBoost seed42 OOF check
# =========================

y_stack = y.astype(int).reset_index(drop=True)

# X_stack은 실험할 데이터로 지정
# 예:
# X_stack = X_te_combo_v2.copy()
# 또는
# X_stack = X_te_combo.copy()

X_stack = X_stack.copy().reset_index(drop=True)

cat_cols = X_stack.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols:
    X_stack[col] = X_stack[col].astype(str)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cat_seed42_oof = np.zeros(len(X_stack))
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
    print(f"\n===== Cat seed42 Fold {fold} =====")

    X_tr = X_stack.iloc[tr_idx].copy()
    X_val = X_stack.iloc[val_idx].copy()
    y_tr = y_stack.iloc[tr_idx]
    y_val = y_stack.iloc[val_idx]

    cat_model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.02498214961001344,
        depth=8,
        l2_leaf_reg=18.591182129683194,
        random_strength=0.32969640414889206,
        bagging_temperature=4.535604806522509,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=100,
        class_weights=[1, 190123 / 66228],
        allow_writing_files=False
    )

    cat_model.fit(
        X_tr,
        y_tr,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=100
    )

    val_proba = cat_model.predict_proba(X_val)[:, 1]
    cat_seed42_oof[val_idx] = val_proba

    fold_auc = roc_auc_score(y_val, val_proba)
    fold_scores.append(fold_auc)

    print(f"Fold {fold} AUC:", fold_auc)

print("\n====================")
print("Fold scores:", fold_scores)
print("Mean AUC:", np.mean(fold_scores))
print("OOF AUC:", roc_auc_score(y_stack, cat_seed42_oof))


===== Cat seed42 Fold 1 =====
0:	test: 0.7265009	best: 0.7265009 (0)	total: 6.21s	remaining: 3h 27m 2s
100:	test: 0.7359457	best: 0.7359457 (100)	total: 57.1s	remaining: 17m 52s
200:	test: 0.7374396	best: 0.7374396 (200)	total: 1m 13s	remaining: 10m 57s
300:	test: 0.7377786	best: 0.7378097 (289)	total: 1m 30s	remaining: 8m 28s
400:	test: 0.7380248	best: 0.7380248 (400)	total: 1m 44s	remaining: 6m 56s
500:	test: 0.7380819	best: 0.7381095 (469)	total: 1m 58s	remaining: 5m 54s
600:	test: 0.7381266	best: 0.7381335 (594)	total: 2m 12s	remaining: 5m 7s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.738133537
bestIteration = 594

Shrink model to first 595 iterations.
Fold 1 AUC: 0.7381335369947317

===== Cat seed42 Fold 2 =====
0:	test: 0.7278318	best: 0.7278318 (0)	total: 166ms	remaining: 5m 32s
100:	test: 0.7384742	best: 0.7384742 (100)	total: 14s	remaining: 4m 22s
200:	test: 0.7410131	best: 0.7410131 (200)	total: 28.3s	remaining: 4m 12s
300:	test: 0.7419238	best: 0.7

In [48]:
final_pred_refined = (
    0.46 * main_cat_rank +
    0.06 * shallow_cat_rank +
    0.36 * random_cat_rank +
    0.12 * xgb_rank
)

submission_refined = submission.copy()
submission_refined[pred_col] = final_pred_refined

submission_refined.to_csv(
    "submission_refined_no_random2.csv",
    index=False
)

## specialist 실험

In [19]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from pathlib import Path

save_dir = Path("../oof_preds/combo_te_s10")

# 기존 v1 OOF 불러오기
combo_cat_oof_seed = np.load(save_dir / "combo_cat_oof_seed.npy")
combo_xgb_oof = np.load(save_dir / "combo_xgb_oof.npy")
combo_cat_shallow_oof = np.load(save_dir / "combo_cat_shallow_oof.npy")
combo_cat_random_oof = np.load(save_dir / "combo_cat_random_oof.npy")

# y_stack
y_stack = np.load("../oof_preds/base_te_s10/y_stack.npy")
y_stack = pd.Series(y_stack).reset_index(drop=True)

# v1 feature set
X_stack = X_te_combo.copy().reset_index(drop=True)

cat_cols = X_stack.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols:
    X_stack[col] = X_stack[col].astype(str)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

main_cat_rank = rankdata(combo_cat_oof_seed) / len(combo_cat_oof_seed)
shallow_cat_rank = rankdata(combo_cat_shallow_oof) / len(combo_cat_shallow_oof)
random_cat_rank = rankdata(combo_cat_random_oof) / len(combo_cat_random_oof)
xgb_rank = rankdata(combo_xgb_oof) / len(combo_xgb_oof)

base_rank = (
    0.44 * main_cat_rank +
    0.08 * shallow_cat_rank +
    0.35 * random_cat_rank +
    0.13 * xgb_rank
)

print("Base OOF AUC:", roc_auc_score(y_stack, base_rank))

Base OOF AUC: 0.7404963277840849


## IVF/DI MASK

In [20]:
def to_num_count(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace("회 이상", "", regex=False)
         .str.replace("회", "", regex=False)
         .str.replace("nan", "0", regex=False),
        errors="coerce"
    ).fillna(0)

ivf_count = to_num_count(X_stack["IVF 시술 횟수"]) if "IVF 시술 횟수" in X_stack.columns else pd.Series(0, index=X_stack.index)
di_count = to_num_count(X_stack["DI 시술 횟수"]) if "DI 시술 횟수" in X_stack.columns else pd.Series(0, index=X_stack.index)

ivf_text_mask = pd.Series(False, index=X_stack.index)
di_text_mask = pd.Series(False, index=X_stack.index)

for col in ["시술 유형", "특정 시술 유형"]:
    if col in X_stack.columns:
        ivf_text_mask |= X_stack[col].astype(str).str.contains("IVF", na=False)
        di_text_mask |= X_stack[col].astype(str).str.contains("DI", na=False)

ivf_mask = ((ivf_count > 0) | ivf_text_mask).to_numpy()
di_mask = ((di_count > 0) | di_text_mask).to_numpy()

print("IVF count:", ivf_mask.sum())
print("DI count:", di_mask.sum())
print("Both IVF & DI:", (ivf_mask & di_mask).sum())
print("Neither:", (~(ivf_mask | di_mask)).sum())

IVF count: 252607
DI count: 15018
Both IVF & DI: 11274
Neither: 0


## Specialist 학습 함수

In [21]:
def train_specialist_oof(
    X_stack,
    y_stack,
    specialist_mask,
    cat_cols,
    skf,
    model_name="specialist",
):
    specialist_oof = np.full(len(X_stack), np.nan)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print(f"\n===== {model_name} Fold {fold} =====")

        tr_idx = np.array(tr_idx)
        val_idx = np.array(val_idx)

        tr_sub_idx = tr_idx[specialist_mask[tr_idx]]
        val_sub_idx = val_idx[specialist_mask[val_idx]]

        print("train subset:", len(tr_sub_idx), "valid subset:", len(val_sub_idx))

        if len(tr_sub_idx) < 100 or len(val_sub_idx) < 20:
            print("Too small subset. skip.")
            continue

        y_tr_sub = y_stack.iloc[tr_sub_idx]
        y_val_sub = y_stack.iloc[val_sub_idx]

        # fold 안에서 클래스가 하나뿐이면 학습/평가 불가
        if y_tr_sub.nunique() < 2 or y_val_sub.nunique() < 2:
            print("Only one class in subset. skip.")
            continue

        X_tr = X_stack.iloc[tr_sub_idx].copy()
        X_val = X_stack.iloc[val_sub_idx].copy()

        model = CatBoostClassifier(
            iterations=2000,
            learning_rate=0.02498214961001344,
            depth=8,
            l2_leaf_reg=18.591182129683194,
            random_strength=0.32969640414889206,
            bagging_temperature=4.535604806522509,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42,
            verbose=100,
            class_weights=[1, 190123 / 66228],
            allow_writing_files=False,
            task_type="GPU",
            devices="0"
        )

        model.fit(
            X_tr,
            y_tr_sub,
            cat_features=cat_cols,
            eval_set=(X_val, y_val_sub),
            early_stopping_rounds=100,
            verbose=100
        )

        pred = model.predict_proba(X_val)[:, 1]
        specialist_oof[val_sub_idx] = pred

        print(
            f"{model_name} Fold {fold} subset AUC:",
            roc_auc_score(y_val_sub, pred)
        )

    filled_mask = ~np.isnan(specialist_oof)
    print(f"\n{model_name} filled:", filled_mask.sum())

    if filled_mask.sum() > 0 and y_stack.iloc[filled_mask].nunique() == 2:
        print(
            f"{model_name} OOF subset AUC:",
            roc_auc_score(y_stack.iloc[filled_mask], specialist_oof[filled_mask])
        )

    return specialist_oof

## specialist oof 생성

In [22]:
ivf_specialist_oof = train_specialist_oof(
    X_stack=X_stack,
    y_stack=y_stack,
    specialist_mask=ivf_mask,
    cat_cols=cat_cols,
    skf=skf,
    model_name="IVF Specialist"
)

di_specialist_oof = train_specialist_oof(
    X_stack=X_stack,
    y_stack=y_stack,
    specialist_mask=di_mask,
    cat_cols=cat_cols,
    skf=skf,
    model_name="DI Specialist"
)


===== IVF Specialist Fold 1 =====
train subset: 202124 valid subset: 50483
0:	test: 0.7271398	best: 0.7271398 (0)	total: 227ms	remaining: 7m 33s
100:	test: 0.7351678	best: 0.7351678 (100)	total: 41.6s	remaining: 13m 1s
200:	test: 0.7365843	best: 0.7365843 (200)	total: 1m 7s	remaining: 10m 6s
300:	test: 0.7369575	best: 0.7369627 (299)	total: 1m 23s	remaining: 7m 51s
400:	test: 0.7371177	best: 0.7371207 (398)	total: 1m 40s	remaining: 6m 39s
500:	test: 0.7372394	best: 0.7372394 (500)	total: 1m 56s	remaining: 5m 48s
600:	test: 0.7372405	best: 0.7372447 (595)	total: 2m 11s	remaining: 5m 6s
700:	test: 0.7372410	best: 0.7372888 (645)	total: 2m 26s	remaining: 4m 31s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7372888173
bestIteration = 645

Shrink model to first 646 iterations.
IVF Specialist Fold 1 subset AUC: 0.7372888173195837

===== IVF Specialist Fold 2 =====
train subset: 202043 valid subset: 50564
0:	test: 0.7273209	best: 0.7273209 (0)	total: 114ms	remaining: 3

In [23]:
spec_dir = Path("../oof_preds/specialist_combo_te_s10")
spec_dir.mkdir(parents=True, exist_ok=True)

np.save(spec_dir / "ivf_specialist_oof.npy", ivf_specialist_oof)
np.save(spec_dir / "di_specialist_oof.npy", di_specialist_oof)
np.save(spec_dir / "ivf_mask.npy", ivf_mask)
np.save(spec_dir / "di_mask.npy", di_mask)

In [24]:
def rank_on_mask(pred, mask):
    out = np.zeros(len(pred))
    valid = mask & ~np.isnan(pred)

    if valid.sum() > 0:
        out[valid] = rankdata(pred[valid]) / valid.sum()

    return out, valid

ivf_spec_rank, ivf_valid = rank_on_mask(ivf_specialist_oof, ivf_mask)
di_spec_rank, di_valid = rank_on_mask(di_specialist_oof, di_mask)

print("IVF valid:", ivf_valid.sum())
print("DI valid:", di_valid.sum())

best_score = 0
best_weights = None

for w_ivf in np.arange(0.00, 0.31, 0.02):
    for w_di in np.arange(0.00, 0.31, 0.02):
        blend = base_rank.copy()

        # IVF row만 보정
        blend[ivf_valid] = (
            (1 - w_ivf) * base_rank[ivf_valid] +
            w_ivf * ivf_spec_rank[ivf_valid]
        )

        # DI row만 보정
        blend[di_valid] = (
            (1 - w_di) * blend[di_valid] +
            w_di * di_spec_rank[di_valid]
        )

        score = roc_auc_score(y_stack, blend)

        if score > best_score:
            best_score = score
            best_weights = (w_ivf, w_di)

print("Specialist blend best_score:", best_score)
print("Specialist blend best_weights:", best_weights)
print("Base score:", roc_auc_score(y_stack, base_rank))

IVF valid: 252607
DI valid: 15018
Specialist blend best_score: 0.7405096553028516
Specialist blend best_weights: (np.float64(0.0), np.float64(0.1))
Base score: 0.7404963277840849


## 최종

In [13]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from category_encoders import TargetEncoder


# =========================================================
# 0. Config
# =========================================================

TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"
SUBMISSION_PATH = "../data/sample_submission.csv"
TARGET = "임신 성공 여부"

FOLD_SEED = 2024
N_SPLITS = 5

POS_WEIGHT = 190123 / 66228

SAVE_DIR = Path("../oof_preds/combo_te_v1_s10_seed2024")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SUB_DIR = Path("../submissions/seed_ensemble")
SUB_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# 1. Utility
# =========================================================

def rank01(pred):
    return rankdata(pred) / len(pred)


def add_combo_columns(X):
    X = X.copy()

    # Combo TE v1 only
    combo_pairs = [
        ("시술 당시 나이", "난자 출처"),
        ("시술 당시 나이", "정자 출처"),
        ("시술 당시 나이", "시술 유형"),
        ("시술 당시 나이", "특정 시술 유형"),
        ("시술 유형", "난자 출처"),
        ("시술 유형", "정자 출처"),
        ("특정 시술 유형", "난자 출처"),
        ("특정 시술 유형", "정자 출처"),
        ("난자 출처", "정자 출처"),
        ("배란 유도 유형", "시술 당시 나이"),
    ]

    combo_cols = []

    for col1, col2 in combo_pairs:
        if col1 in X.columns and col2 in X.columns:
            new_col = f"{col1}_{col2}_combo"
            X[new_col] = X[col1].astype(str) + "_" + X[col2].astype(str)
            combo_cols.append(new_col)

    return X, combo_cols


def add_oof_target_encoding(X, y, cols, n_splits=5, smoothing=10, random_state=2024):
    X = X.copy().reset_index(drop=True)
    y = y.astype(int).reset_index(drop=True)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    te_features = pd.DataFrame(index=X.index)

    for col in cols:
        te_features[f"{col}_TE"] = 0.0

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"Target Encoding Fold {fold}")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_tr = y.iloc[train_idx]

        encoder = TargetEncoder(
            cols=cols,
            smoothing=smoothing
        )

        encoder.fit(X_tr[cols], y_tr)
        encoded_val = encoder.transform(X_val[cols])

        for col in cols:
            te_features.loc[val_idx, f"{col}_TE"] = encoded_val[col].values

    final_encoder = TargetEncoder(
        cols=cols,
        smoothing=smoothing
    )

    final_encoder.fit(X[cols], y)

    X_te = pd.concat([X, te_features], axis=1)

    return X_te, final_encoder


def make_cat_params(kind="main", seed=42):
    if kind == "main":
        return dict(
            iterations=2000,
            learning_rate=0.02498214961001344,
            depth=8,
            l2_leaf_reg=18.591182129683194,
            random_strength=0.32969640414889206,
            bagging_temperature=4.535604806522509,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    if kind == "shallow":
        return dict(
            iterations=2500,
            learning_rate=0.02,
            depth=5,
            l2_leaf_reg=8,
            random_strength=1.5,
            bagging_temperature=2,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    if kind == "random":
        return dict(
            iterations=2500,
            learning_rate=0.022,
            depth=7,
            l2_leaf_reg=15,
            random_strength=8,
            bagging_temperature=8,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    raise ValueError(f"Unknown CatBoost kind: {kind}")


# =========================================================
# 2. Load & preprocess
# =========================================================

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
submission = pd.read_csv(SUBMISSION_PATH)

X_raw = train_raw.drop(columns=[TARGET])
y = train_raw[TARGET].astype(int).reset_index(drop=True)
X_test_raw = test_raw.copy()

id_cols = [col for col in X_raw.columns if "ID" in col.upper()]

X_raw = X_raw.drop(columns=id_cols, errors="ignore")
X_test_raw = X_test_raw.drop(columns=id_cols, errors="ignore")

# data_preprocessing 함수는 위 셀에 정의되어 있어야 함
X = data_preprocessing(X_raw)
X_test = data_preprocessing(X_test_raw)

X_combo, combo_cols = add_combo_columns(X)
X_test_combo, _ = add_combo_columns(X_test)

base_te_cols = [
    "시술 시기 코드",
    "시술 유형",
    "특정 시술 유형",
    "배란 유도 유형",
    "난자 출처",
    "정자 출처",
    "배아 생성 주요 이유",
    "시술 당시 나이",
]

base_te_cols = [col for col in base_te_cols if col in X_combo.columns]
te_cols = base_te_cols + combo_cols

print("base_te_cols:", base_te_cols)
print("combo_cols:", combo_cols)
print("te_cols count:", len(te_cols))

X_te_combo, te_encoder_combo = add_oof_target_encoding(
    X_combo,
    y,
    cols=te_cols,
    n_splits=N_SPLITS,
    smoothing=10,
    random_state=FOLD_SEED
)

test_te_values = te_encoder_combo.transform(X_test_combo[te_cols])

for col in te_cols:
    X_test_combo[f"{col}_TE"] = test_te_values[col].values

# combo 문자열 원본 제거
X_te_combo = X_te_combo.drop(columns=combo_cols, errors="ignore")
X_test_te_combo = X_test_combo.drop(columns=combo_cols, errors="ignore")

# 컬럼 정렬
X_test_te_combo = X_test_te_combo[X_te_combo.columns]

X_stack = X_te_combo.copy().reset_index(drop=True)
X_test_stack = X_test_te_combo.copy().reset_index(drop=True)
y_stack = y.reset_index(drop=True)

cat_cols = X_stack.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols:
    X_stack[col] = X_stack[col].astype(str)
    X_test_stack[col] = X_test_stack[col].astype(str)

print("X_stack:", X_stack.shape)
print("X_test_stack:", X_test_stack.shape)
print("y_stack:", y_stack.shape)
print("cat_cols:", len(cat_cols))
print("combo cols remaining:", [c for c in X_stack.columns if c.endswith("_combo")])

assert list(X_stack.columns) == list(X_test_stack.columns)
assert len(X_stack) == len(y_stack)
assert len(X_test_stack) == len(submission)

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=FOLD_SEED
)


# =========================================================
# 3. Train CatBoost seed ensemble: OOF + test
# =========================================================

def train_cat_seed_ensemble(
    X_stack,
    X_test_stack,
    y_stack,
    cat_cols,
    skf,
    seeds,
    kind="main",
    name="main_cat"
):
    seed_oof_list = []
    seed_test_list = []
    score_rows = []

    for seed in seeds:
        print(f"\n================ {name} seed {seed} ================")

        oof = np.zeros(len(X_stack))
        test_pred = np.zeros(len(X_test_stack))

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
            print(f"\n{name} seed {seed} / Fold {fold}")

            X_tr = X_stack.iloc[tr_idx].copy()
            X_val = X_stack.iloc[val_idx].copy()
            y_tr = y_stack.iloc[tr_idx]
            y_val = y_stack.iloc[val_idx]

            model = CatBoostClassifier(**make_cat_params(kind=kind, seed=seed))

            model.fit(
                X_tr,
                y_tr,
                cat_features=cat_cols,
                eval_set=(X_val, y_val),
                early_stopping_rounds=100 if kind == "main" else 150,
                verbose=100
            )

            val_pred = model.predict_proba(X_val)[:, 1]
            oof[val_idx] = val_pred

            fold_auc = roc_auc_score(y_val, val_pred)
            print(f"{name} seed {seed} Fold {fold} AUC:", fold_auc)

            score_rows.append({
                "model": name,
                "seed": seed,
                "fold": fold,
                "auc": fold_auc
            })

            test_pred += model.predict_proba(X_test_stack)[:, 1] / skf.n_splits

        seed_auc = roc_auc_score(y_stack, oof)
        print(f"\n{name} seed {seed} OOF AUC:", seed_auc)

        score_rows.append({
            "model": name,
            "seed": seed,
            "fold": "OOF",
            "auc": seed_auc
        })

        seed_oof_list.append(oof)
        seed_test_list.append(test_pred)

        np.save(SAVE_DIR / f"{name}_seed{seed}_oof.npy", oof)
        np.save(SAVE_DIR / f"{name}_seed{seed}_test.npy", test_pred)

    final_oof = np.mean(seed_oof_list, axis=0)
    final_test = np.mean(seed_test_list, axis=0)

    final_auc = roc_auc_score(y_stack, final_oof)
    print(f"\n{name} seed ensemble OOF AUC:", final_auc)

    score_rows.append({
        "model": name,
        "seed": "ensemble",
        "fold": "OOF",
        "auc": final_auc
    })

    return final_oof, final_test, pd.DataFrame(score_rows)


def train_cat_single(
    X_stack,
    X_test_stack,
    y_stack,
    cat_cols,
    skf,
    kind="shallow",
    name="shallow_cat"
):
    oof = np.zeros(len(X_stack))
    test_pred = np.zeros(len(X_test_stack))
    score_rows = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print(f"\n================ {name} Fold {fold} ================")

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        model = CatBoostClassifier(**make_cat_params(kind=kind, seed=42))

        early_stop = 150 if kind == "shallow" else 200

        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_cols,
            eval_set=(X_val, y_val),
            early_stopping_rounds=early_stop,
            verbose=100
        )

        val_pred = model.predict_proba(X_val)[:, 1]
        oof[val_idx] = val_pred

        fold_auc = roc_auc_score(y_val, val_pred)
        print(f"{name} Fold {fold} AUC:", fold_auc)

        score_rows.append({
            "model": name,
            "seed": 42,
            "fold": fold,
            "auc": fold_auc
        })

        test_pred += model.predict_proba(X_test_stack)[:, 1] / skf.n_splits

        # 중간 체크포인트
        np.save(SAVE_DIR / f"{name}_partial_oof_fold{fold}.npy", oof)
        np.save(SAVE_DIR / f"{name}_partial_test_fold{fold}.npy", test_pred)

    final_auc = roc_auc_score(y_stack, oof)
    print(f"\n{name} OOF AUC:", final_auc)

    score_rows.append({
        "model": name,
        "seed": 42,
        "fold": "OOF",
        "auc": final_auc
    })

    return oof, test_pred, pd.DataFrame(score_rows)


# Main Cat seed ensemble
main_cat_oof, main_cat_test_pred, main_score_df = train_cat_seed_ensemble(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    seeds=[42, 77, 2024],
    kind="main",
    name="main_cat"
)

np.save(SAVE_DIR / "main_cat_oof.npy", main_cat_oof)
np.save(SAVE_DIR / "main_cat_test_pred.npy", main_cat_test_pred)


# Shallow Cat
shallow_cat_oof, shallow_cat_test_pred, shallow_score_df = train_cat_single(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    kind="shallow",
    name="shallow_cat"
)

np.save(SAVE_DIR / "shallow_cat_oof.npy", shallow_cat_oof)
np.save(SAVE_DIR / "shallow_cat_test_pred.npy", shallow_cat_test_pred)


# Random Cat
random_cat_oof, random_cat_test_pred, random_score_df = train_cat_single(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    kind="random",
    name="random_cat"
)

np.save(SAVE_DIR / "random_cat_oof.npy", random_cat_oof)
np.save(SAVE_DIR / "random_cat_test_pred.npy", random_cat_test_pred)


# =========================================================
# 4. XGB: OOF + test
# =========================================================

def train_xgb_oof_test(X_stack, X_test_stack, y_stack, skf):
    numeric_features = X_stack.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_stack.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])

    oof = np.zeros(len(X_stack))
    test_pred = np.zeros(len(X_test_stack))
    score_rows = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print(f"\n================ XGB Fold {fold} ================")

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        X_tr_trans = preprocessor.fit_transform(X_tr)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test_stack)

        model = XGBClassifier(
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="auc",
            random_state=42,
            scale_pos_weight=POS_WEIGHT,
            tree_method="hist",
            n_jobs=-1
        )

        model.fit(
            X_tr_trans,
            y_tr.to_numpy().ravel(),
            eval_set=[(X_val_trans, y_val.to_numpy().ravel())],
            verbose=False
        )

        val_pred = model.predict_proba(X_val_trans)[:, 1]
        oof[val_idx] = val_pred

        fold_auc = roc_auc_score(y_val, val_pred)
        print("XGB Fold AUC:", fold_auc)

        score_rows.append({
            "model": "xgb",
            "seed": 42,
            "fold": fold,
            "auc": fold_auc
        })

        test_pred += model.predict_proba(X_test_trans)[:, 1] / skf.n_splits

        np.save(SAVE_DIR / f"xgb_partial_oof_fold{fold}.npy", oof)
        np.save(SAVE_DIR / f"xgb_partial_test_fold{fold}.npy", test_pred)

    final_auc = roc_auc_score(y_stack, oof)
    print("\nXGB OOF AUC:", final_auc)

    score_rows.append({
        "model": "xgb",
        "seed": 42,
        "fold": "OOF",
        "auc": final_auc
    })

    return oof, test_pred, pd.DataFrame(score_rows)


xgb_oof, xgb_test_pred, xgb_score_df = train_xgb_oof_test(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    skf=skf
)

np.save(SAVE_DIR / "xgb_oof.npy", xgb_oof)
np.save(SAVE_DIR / "xgb_test_pred.npy", xgb_test_pred)


# =========================================================
# 5. OOF validation: rank blend
# =========================================================

main_cat_rank_oof = rank01(main_cat_oof)
shallow_cat_rank_oof = rank01(shallow_cat_oof)
random_cat_rank_oof = rank01(random_cat_oof)
xgb_rank_oof = rank01(xgb_oof)

final_oof_seed2024 = (
    0.44 * main_cat_rank_oof +
    0.08 * shallow_cat_rank_oof +
    0.35 * random_cat_rank_oof +
    0.13 * xgb_rank_oof
)

final_oof_auc = roc_auc_score(y_stack, final_oof_seed2024)

print("\n====================")
print("Seed2024 Main Cat OOF:", roc_auc_score(y_stack, main_cat_oof))
print("Seed2024 Shallow Cat OOF:", roc_auc_score(y_stack, shallow_cat_oof))
print("Seed2024 Random Cat OOF:", roc_auc_score(y_stack, random_cat_oof))
print("Seed2024 XGB OOF:", roc_auc_score(y_stack, xgb_oof))
print("Seed2024 Final Rank Blend OOF:", final_oof_auc)

np.save(SAVE_DIR / "final_oof_seed2024.npy", final_oof_seed2024)
np.save(SAVE_DIR / "y_stack.npy", y_stack.to_numpy())


# score log 저장
score_df = pd.concat(
    [main_score_df, shallow_score_df, random_score_df, xgb_score_df],
    ignore_index=True
)

score_df.to_csv(SAVE_DIR / "fold_oof_scores.csv", index=False)

summary_df = pd.DataFrame([
    {
        "fold_seed": FOLD_SEED,
        "feature_set": "combo_te_v1_s10",
        "main_cat_oof": roc_auc_score(y_stack, main_cat_oof),
        "shallow_cat_oof": roc_auc_score(y_stack, shallow_cat_oof),
        "random_cat_oof": roc_auc_score(y_stack, random_cat_oof),
        "xgb_oof": roc_auc_score(y_stack, xgb_oof),
        "final_rank_blend_oof": final_oof_auc,
        "weights": "main 0.44 / shallow 0.08 / random 0.35 / xgb 0.13"
    }
])

summary_df.to_csv(SAVE_DIR / "summary.csv", index=False)
display(summary_df)


# =========================================================
# 6. Test prediction: seed2024 submission
# =========================================================

main_cat_rank_test = rank01(main_cat_test_pred)
shallow_cat_rank_test = rank01(shallow_cat_test_pred)
random_cat_rank_test = rank01(random_cat_test_pred)
xgb_rank_test = rank01(xgb_test_pred)

final_pred_seed2024 = (
    0.44 * main_cat_rank_test +
    0.08 * shallow_cat_rank_test +
    0.35 * random_cat_rank_test +
    0.13 * xgb_rank_test
)

assert len(final_pred_seed2024) == len(submission)
assert np.isfinite(final_pred_seed2024).all()
assert final_pred_seed2024.min() >= 0
assert final_pred_seed2024.max() <= 1

pred_col = submission.columns[-1]

submission_seed2024 = submission.copy()
submission_seed2024[pred_col] = final_pred_seed2024

submission_seed2024.to_csv(
    SUB_DIR / "submission_combo_te_v1_rank_seed2024.csv",
    index=False
)

np.save(
    SUB_DIR / "final_pred_combo_te_v1_seed2024.npy",
    final_pred_seed2024
)

print("\nseed2024 저장 완료")
print(submission_seed2024[pred_col].describe())


# =========================================================
# 7. seed42 + seed2024 average candidate
# =========================================================

seed42_path = Path("../submissions/preds/final_aggressive_pred_lb_0_74172.npy")

if seed42_path.exists():
    final_pred_seed42 = np.load(seed42_path)

    assert len(final_pred_seed42) == len(final_pred_seed2024)

    final_pred_seed42_2024_avg = (
        0.5 * final_pred_seed42 +
        0.5 * final_pred_seed2024
    )

    assert np.isfinite(final_pred_seed42_2024_avg).all()
    assert final_pred_seed42_2024_avg.min() >= 0
    assert final_pred_seed42_2024_avg.max() <= 1

    submission_avg = submission.copy()
    submission_avg[pred_col] = final_pred_seed42_2024_avg

    submission_avg.to_csv(
        SUB_DIR / "submission_combo_te_v1_rank_seed42_2024_avg.csv",
        index=False
    )

    np.save(
        SUB_DIR / "final_pred_combo_te_v1_seed42_2024_avg.npy",
        final_pred_seed42_2024_avg
    )

    print("\nseed42 + seed2024 평균 저장 완료")
    print(submission_avg[pred_col].describe())

else:
    print("\nseed42 npy 파일이 없어서 평균 파일은 생성하지 않았습니다.")
    print("찾은 경로:", seed42_path)

base_te_cols: ['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '난자 출처', '정자 출처', '배아 생성 주요 이유', '시술 당시 나이']
combo_cols: ['시술 당시 나이_난자 출처_combo', '시술 당시 나이_정자 출처_combo', '시술 당시 나이_시술 유형_combo', '시술 당시 나이_특정 시술 유형_combo', '시술 유형_난자 출처_combo', '시술 유형_정자 출처_combo', '특정 시술 유형_난자 출처_combo', '특정 시술 유형_정자 출처_combo', '난자 출처_정자 출처_combo', '배란 유도 유형_시술 당시 나이_combo']
te_cols count: 18
Target Encoding Fold 1
Target Encoding Fold 2
Target Encoding Fold 3
Target Encoding Fold 4
Target Encoding Fold 5
X_stack: (256351, 110)
X_test_stack: (90067, 110)
y_stack: (256351,)
cat_cols: 54
combo cols remaining: []

================ main_cat seed 42 ================

main_cat seed 42 / Fold 1
0:	test: 0.7322805	best: 0.7322805 (0)	total: 4.32s	remaining: 2h 23m 50s
100:	test: 0.7398837	best: 0.7398837 (100)	total: 33.4s	remaining: 10m 28s
200:	test: 0.7418165	best: 0.7418165 (200)	total: 48.5s	remaining: 7m 14s
300:	test: 0.7423926	best: 0.7423926 (300)	total: 1m 1s	remaining: 5m 46s
400:	test: 0.7426074	best: 0.

,fold_seed,feature_set,main_cat_oof,shallow_cat_oof,random_cat_oof,xgb_oof,final_rank_blend_oof,weights
0,2024,combo_te_v1_s10,0.740497,0.740181,0.740416,0.738223,0.740568,main 0.44 / shallow 0.08 / random 0.35 / xgb 0.13



seed2024 저장 완료
count    90067.000000
mean         0.500006
std          0.288347
min          0.000017
25%          0.250642
50%          0.500188
75%          0.749656
max          0.999988
Name: probability, dtype: float64

seed42 + seed2024 평균 저장 완료
count    90067.000000
mean         0.500006
std          0.288312
min          0.000024
25%          0.250951
50%          0.500341
75%          0.749503
max          0.999989
Name: probability, dtype: float64


In [2]:

final_pred_seed42 = np.load("../submissions/preds/final_aggressive_pred_lb_0_74172.npy")
final_pred_seed_avg = (
    0.5 * final_pred_seed42 +
    0.5 * final_pred_seed2024
)

submission_avg = submission.copy()
submission_avg[pred_col] = final_pred_seed_avg

submission_avg.to_csv(
    "submission_combo_te_v1_rank_seed42_2024_avg.csv",
    index=False
)
np.save(
    "final_pred_combo_te_v1_seed42_2024_avg.npy",
    final_pred_seed_avg
)
print("seed42 + seed2024 평균 저장 완료")
print(submission_avg[pred_col].describe())


In [15]:
seed42_oof = np.load("../oof_preds/combo_te_s10/final_combo_rank.npy")
seed2024_oof = np.load("../oof_preds/combo_te_v1_s10_seed2024/final_oof_seed2024.npy")

avg_oof = 0.5 * seed42_oof + 0.5 * seed2024_oof

roc_auc_score(y_stack, avg_oof)

0.7407129516061617

In [16]:
from pathlib import Path

for p in [
    "submissions/seed_ensemble/submission_combo_te_v1_rank_seed42_2024_avg.csv",
    "submissions/seed_ensemble/submission_combo_te_v1_rank_seed2024.csv",
]:
    path = Path(p)
    print(path, path.exists(), path.stat().st_size if path.exists() else None)

submissions/seed_ensemble/submission_combo_te_v1_rank_seed42_2024_avg.csv True 2725597
submissions/seed_ensemble/submission_combo_te_v1_rank_seed2024.csv True 2725732
